In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case)

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/11120
11120


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

In [7]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [8]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [9]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1':
    cntrl_vars_init = [1]
elif case[3] == '2':
    cntrl_vars_init = [0,1]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [10]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_' + case + '.pickle'
case_1 = case[0] + case[1] + '0' + case[3] + case[4]
final_file_1 = 'control_' + case_1 + '.pickle'

In [11]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

In [12]:
# get initial parameters and target states

i_stepsize = 5
i_range = range(0, len(exc),i_stepsize)
i_range_0 = range(0, len(exc),i_stepsize)
i_range_1 = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.750000000000000

In [13]:
# get uncontrolled cost

data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  0 , total integrated cost =  5097.289828199723
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002

In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    
    if not type(bestControl_init[i]) == type(None):
        continue
        
    control0 = aln.getZeroControl()

    ##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)

    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    with open(init_file,'wb') as f:
        pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  1 , total integrated cost =  55.886170914069865
RUN  2 , total integrated cost =  42.65570180421014
RUN  3 , total integrated cost =  28.840267049881813
RUN  4 , total integrated cost =  24.406095908721905
RUN  5 , total integrated cost =  20.071541794363235
RUN  6 , total integrated cost =  17.497077501812623
RUN  7 , total integrated cost =  14.962898550512925
RUN  8 , total integrated cost =  12.48803072824853
RUN  9 , total integrated cost =  10.253848886718844
RUN  10 , total integrated cost =  8.019764971048419
RUN  11 , total integrated cost =  7.377795308952696
RUN  12 , total integrated cost =  5.786539815314411
RUN  13 , total integrated cost =  5.780689329820276
RUN  14 , total integrated cost =  5.600426059938204
RUN  15 , total integrated cost =  5.494742548840481
RUN  1

ERROR:root:Problem in initial value trasfer


RUN  70 , total integrated cost =  4.447417820305347
Control only changes marginally.
RUN  70 , total integrated cost =  4.447417820305347
Improved over  70  iterations in  34.089943308383226  seconds by  99.92465077022484  percent.
Problem in initial value trasfer:  Vmean_exc -64.16262123696094 -64.15379235761353
weight =  13271.535793848892
set cost params:  1.0 13271.535793848892 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5548.689928220025
Gradient descend method:  None
RUN  1 , total integrated cost =  5091.108782491913
RUN  2 , total integrated cost =  5078.348427359097
RUN  3 , total integrated cost =  5045.199922119573
RUN  4 , total integrated cost =  5032.103811287828
RUN  5 , total integrated cost =  5032.091594644799
RUN  6 , total integrated cost =  5032.091216718317
RUN  7 , total integrated cost =  5032.0912052717795
RUN  8 , total integrated cost =  5032.091202760467
RUN  9 , total integrated cost =  5032.091201845418
RUN  10 , total inte

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  24 , total integrated cost =  5032.09120122484
Improved over  24  iterations in  4.418189903721213  seconds by  9.310282853756533  percent.
Problem in initial value trasfer:  Vmean_exc -60.58104226392273 -60.609701692552676
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  1 , total integrated cost =  33.4728633380101
RUN  2 , total integrated cost =  24.481962647867114
RUN  3 , total integrated cost =  12.252280206360792
RUN  4 , total integrated cost =  10.864250324454556
RUN  5 , total integrated cost =  9.457756214128958
RUN  6 , total integrated cost =  8.483810970984504
RUN  7 , total integrated cost =  7.46490663627272
RUN  8 , total integrated cost =  6.621770016958646
RUN  9 , total integrated cost =  5.560869975790674
RUN  10 , total integrated cost =  4.967303149522364
RUN  11 , tota

...

Control only changes marginally.
RUN  25 , total integrated cost =  9125.414573374012
Improved over  25  iterations in  3.7397734988480806  seconds by  7.813865488628977  percent.
Problem in initial value trasfer:  Vmean_exc -61.22560698815414 -61.26939509012557
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33290.05146687772
Gradient descend method:  None
RUN  1 , total integrated cost =  778.7788659264161
RUN  2 , total integrated cost =  506.3550952371859
RUN  3 , total integrated cost =  338.83700764781366
RUN  4 , total integrated cost =  282.1136978199781
RUN  5 , total integrated cost =  240.82597879662492
RUN  6 , total integrated cost =  203.33803375809362
RUN  7 , total integrated cost =  188.9839478770172
RUN  8 , total integrated cost =  184.66602353156415
RUN  9 , total integrated cost =  181.2473204043028
RUN  10 , total integrated cost =  180.80432392003593
RUN  11 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  68 , total integrated cost =  156.3293145822981
Improved over  68  iterations in  8.700841534882784  seconds by  99.53040230431053  percent.
Problem in initial value trasfer:  Vmean_exc -62.0367577175908 -62.04886363423715
weight =  2129.482340265266
set cost params:  1.0 2129.482340265266 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  30587.873190850813
Gradient descend method:  None
RUN  1 , total integrated cost =  29719.974834924033
RUN  2 , total integrated cost =  20512.108034627145
RUN  3 , total integrated cost =  20024.8491771562
RUN  4 , total integrated cost =  19923.94692516088
RUN  5 , total integrated cost =  19923.946925160868


ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  19923.946925160868
Control only changes marginally.
RUN  6 , total integrated cost =  19923.946925160868
Improved over  6  iterations in  1.1540427412837744  seconds by  34.86324857943916  percent.
Problem in initial value trasfer:  Vmean_exc -56.687436003782594 -56.689886926805656


In [15]:
"""
#plot initial guesses
for i in i_range:
    print("---------", i)
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
    plt.show()
"""

'\n#plot initial guesses\nfor i in i_range:\n    print("---------", i)\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n    plt.show()\n'

In [16]:
found_solution = []
no_solution = []
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break


    for i in i_range:
        print("------- ", i, exc[i], inh[i])        

        if np.abs(np.mean(bestState_init[i][0,0,-300:]) - target[i][0,0,-1]) < 0.1 * np.abs(
            np.mean(bestState_init[i][0,0,-100:]) - bestState_init[i][0,0,0]) and np.abs(
            np.mean(bestState_init[i][0,1,-100:]) - target[i][0,1,-1]) < 0.5 * np.abs(
            np.mean(bestState_init[i][0,1,-100:]) - bestState_init[i][0,1,0]) and np.amin(
            bestState_init[i][0,0,:]) > target[i][0,0,-1] - 5. and np.amin(
            bestState_init[i][0,1,:]) > target[i][0,1,-1] - 5.:
            # and np.amin(bestState_init[i][0,0,:]) > bestState_init[i][0,0,0] - 1.
            #and np.amin(bestState_init[i][0,1,:]) > bestState_init[i][0,1,0] - 1.:
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

            
        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        with open(init_file,'wb') as f:
            pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                         costnode_init, weights_init], f)

------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  5 0.4000000000000001 0.40000000000000013
found solution for  5
-------  10 0.4250000000000001 0.42500000000000016
[0, 5] []
closest index  5
set cost params:  1.0 10.0 0.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8953.748973940988
Gradient descend method:  None
RUN  1 , total integrated cost =  126.82485977934402
RUN  2 , total integrated cost =  94.65655009409147
RUN  3 , total integrated cost =  65.98694039678993
RUN  4 , total integrated cost =  54.17276949521651
RUN  5 , total integrated cost =  44.36568684863811
RUN  6 , total integrated cost =  38.47376548794586
RUN  7 , total integrated cost =  34.133868994300926
RUN  8 , total integrated cost =  29.25004343086263
RUN  

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  48 , total integrated cost =  12.58715448129742
Improved over  48  iterations in  6.078982973471284  seconds by  99.85942028844084  percent.
Problem in initial value trasfer:  Vmean_exc -67.37269981229633 -67.38194472628766
weight =  7238.694419576026
set cost params:  1.0 7238.694419576026 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8925.185666131281
Gradient descend method:  None
RUN  1 , total integrated cost =  8034.55606666011
RUN  2 , total integrated cost =  8028.229226753057
RUN  3 , total integrated cost =  8027.674394157972
RUN  4 , total integrated cost =  8027.6583991122125
RUN  5 , total integrated cost =  8027.657419307958
RUN  6 , total integrated cost =  8027.65738082586
RUN  7 , total integrated cost =  8027.657377732015
RUN  8 , total integrated cost =  8027.65737742339
RUN  9 , total integrated cost =  8027.6573773906375
RUN  10 , total integrated cost =  8027.6573773871705
RUN  11 , total integrat

ERROR:root:Problem in initial value trasfer


 percent.
Problem in initial value trasfer:  Vmean_exc -59.57537560608701 -59.59734955719297
-------  15 0.4500000000000001 0.4500000000000002
[0, 5] []
closest index  5
set cost params:  1.0 10.0 0.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  12898.923554004316
Gradient descend method:  None
RUN  1 , total integrated cost =  242.95561016160264
RUN  2 , total integrated cost =  170.93987456504237
RUN  3 , total integrated cost =  107.5426781898293
RUN  4 , total integrated cost =  88.3014977743853
RUN  5 , total integrated cost =  75.28401064240164
RUN  6 , total integrated cost =  66.26689774577147
RUN  7 , total integrated cost =  60.62646923522825
RUN  8 , total integrated cost =  52.536881887806665
RUN  9 , total integrated cost =  49.73789198547407
RUN  10 , total integrated cost =  42.1299467116143
RUN  11 , total integrated cost =  41.429120803078504
RUN  12 , total integrated cost =  41.2022128535257
RUN  13 , total integrated 

...

RUN  7 , total integrated cost =  14416.648684617696
Control only changes marginally.
RUN  7 , total integrated cost =  14416.648684617696
Improved over  7  iterations in  1.8114411756396294  seconds by  34.96924073620791  percent.
Problem in initial value trasfer:  Vmean_exc -56.66635665571716 -56.668528304252696
-------  140 0.4500000000000001 0.9000000000000006
-------  145 0.5750000000000002 0.9000000000000006
[0, 5, 25, 30, 55, 115, 140, 10, 100, 70, 15, 20, 50, 85, 125] [140, 115, 55, 100, 70, 30, 25, 15, 125, 85, 50, 20, 10, 5]
closest index  0
set cost params:  1.0 10.0 0.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33393.651367989325
Gradient descend method:  None
RUN  1 , total integrated cost =  778.1611799449825
RUN  2 , total integrated cost =  508.4037834546732
RUN  3 , total integrated cost =  339.78197716252595
RUN  4 , total integrated cost =  282.8146924057145
RUN  5 , total integrated cost =  241.259682090414
RUN  6 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  57 , total integrated cost =  156.35215916010884
Improved over  57  iterations in  7.0352007281035185  seconds by  99.53179076634314  percent.
Problem in initial value trasfer:  Vmean_exc -62.03332597538171 -62.045399049244324
weight =  2129.1712020930777
set cost params:  1.0 2129.1712020930777 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  30585.653246238744
Gradient descend method:  None
RUN  1 , total integrated cost =  25749.77435798073
RUN  2 , total integrated cost =  24936.345549963542
RUN  3 , total integrated cost =  24719.09049560048
RUN  4 , total integrated cost =  23802.557175199076
RUN  5 , total integrated cost =  23139.194446419555
RUN  6 , total integrated cost =  20297.109596589366
RUN  7 , total integrated cost =  19977.77321113352
RUN  8 , total integrated cost =  19922.614873890314
RUN  9 , total integrated cost =  19922.614262563613
RUN  10 , total integrated cost =  19922.614262328214
RUN  11 , t

ERROR:root:Problem in initial value trasfer


RUN  14 , total integrated cost =  19922.6142623282
Control only changes marginally.
RUN  14 , total integrated cost =  19922.6142623282
Improved over  14  iterations in  2.2502349596470594  seconds by  34.862878023446584  percent.
Problem in initial value trasfer:  Vmean_exc -56.68967277711363 -56.6918655271954
------------------------------------------------------------
-------------------- 15
------------------------------------------------------------
found solution:  [0, 5, 25, 30, 55, 115, 140, 10, 100, 70, 15, 20, 50, 85, 125]
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
closest index  -1
set cost params:  1.0 10.

In [ ]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print("------------------------------------------------")
    print('-------------------------', counter)
    
    if counter > 20:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        
        if conv_init[i] == [True, True]:
            continue
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
                       
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        with open(init_file,'wb') as f:
            pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                         costnode_init, weights_init], f)
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
    counter += 1

------------------------------------------------
------------------------- 0
[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
weight =  15565.887746388795
set cost params:  1.0 15565.887746388795 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5886.263797013163
Gradient descend method:  None
RUN  1 , total integrated cost =  5430.341451743033
RUN  2 , total integrated cost =  4781.27397409858
RUN  3 , total integrated cost =  4770.603432511596
RUN  4 , total integrated cost =  4769.6921

ERROR:root:Problem in initial value trasfer


RUN  10 , total integrated cost =  4766.5964198979955
Control only changes marginally.
RUN  10 , total integrated cost =  4766.5964198979955
Improved over  10  iterations in  2.6401555966585875  seconds by  19.021698920176064  percent.
Problem in initial value trasfer:  Vmean_exc -56.62856989939461 -56.62849698834292
no convergence
-------  5 0.4000000000000001 0.40000000000000013
weight =  26575.349302779934
set cost params:  1.0 26575.349302779934 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5077.20577342451
Gradient descend method:  None
RUN  1 , total integrated cost =  5075.659775710029
RUN  2 , total integrated cost =  5075.659589657514
RUN  3 , total integrated cost =  5075.659589164091
RUN  4 , total integrated cost =  5075.659589164079
RUN  5 , total integrated cost =  5075.659589164076
RUN  6 , total integrated cost =  5075.659589164073
RUN  7 , total integrated cost =  5075.65958916407
RUN  8 , total integrated cost =  5075.659589164063


ERROR:root:Problem in initial value trasfer


RUN  9 , total integrated cost =  5075.659589164063
Control only changes marginally.
RUN  9 , total integrated cost =  5075.659589164063
Improved over  9  iterations in  2.319718884304166  seconds by  0.030453448795412896  percent.
Problem in initial value trasfer:  Vmean_exc -60.75538733951406 -60.79281759615522
no convergence
-------  10 0.4250000000000001 0.42500000000000016
weight =  8214.977108800013
set cost params:  1.0 8214.977108800013 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9091.449498558442
Gradient descend method:  None
RUN  1 , total integrated cost =  8853.343983697756
RUN  2 , total integrated cost =  6875.504924136277
RUN  3 , total integrated cost =  6843.768905557902
RUN  4 , total integrated cost =  6842.32891437291
RUN  5 , total integrated cost =  6838.6655148767095
RUN  6 , total integrated cost =  6837.2070167776565
RUN  7 , total integrated cost =  6837.206809374837
RUN  8 , total integrated cost =  6837.206809374833
RUN  9 , 

...

RUN  5 , total integrated cost =  8230.773429806108
Control only changes marginally.
RUN  5 , total integrated cost =  8230.773429806108
Improved over  5  iterations in  1.50481659732759  seconds by  2.7828117765693605e-06  percent.
Problem in initial value trasfer:  Vmean_exc -60.16665995065525 -60.19773200900591
no convergence
-------  30 0.4250000000000001 0.5250000000000002
weight =  11185.617361812567
set cost params:  1.0 11185.617361812567 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  7977.470663647499
Gradient descend method:  None
RUN  1 , total integrated cost =  7977.470629134782
RUN  2 , total integrated cost =  7977.470628541727
RUN  3 , total integrated cost =  7977.470628472005
RUN  4 , total integrated cost =  7977.470628459574
RUN  5 , total integrated cost =  7977.470628457459
RUN  6 , total integrated cost =  7977.470628457006
RUN  7 , total integrated cost =  7977.470628456895
RUN  8 , total integrated cost =  7977.470628456845
RUN  9 ,

ERROR:root:Problem in initial value trasfer


RUN  11 , total integrated cost =  7977.470628456829
Control only changes marginally.
RUN  11 , total integrated cost =  7977.470628456829
Improved over  11  iterations in  2.184457005932927  seconds by  4.411256639968997e-07  percent.
Problem in initial value trasfer:  Vmean_exc -61.36329598520064 -61.40658510934222
no convergence
-------  35 0.5500000000000003 0.5250000000000002
weight =  5023.32293585167
set cost params:  1.0 5023.32293585167 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  24780.841502518313
Gradient descend method:  None
RUN  1 , total integrated cost =  24207.356637167784
RUN  2 , total integrated cost =  24207.356486274883
RUN  3 , total integrated cost =  24207.356486274875


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  24207.356486274875
Control only changes marginally.
RUN  4 , total integrated cost =  24207.356486274875
Improved over  4  iterations in  1.2611179687082767  seconds by  2.3142273686918884  percent.
Problem in initial value trasfer:  Vmean_exc -56.70091566940655 -56.70150835840969


In [ ]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

In [ ]:
factor_iteration = 20
conv_0 = [[False]*2] * len(exc)
full_converge = False

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 20:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break
        
    counter += 1
    
    for i in i_range_0:
        print("------- ", i, exc[i], inh[i])
        
        if conv_0[i] == [True, True]:
            continue
            
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

    # exc and inh control current 

        setinit(initVars[i], aln)
        aln.params.duration = dur

        if not type(bestControl_0[i]) == type(None):
            control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
        else:
            control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
            weights_0[i] = weights_init[i]
            cost_0[i] = cost_init[i]

        cgv = None
        max_it = 500 * factor_iteration

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                           + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        with open(final_file,'wb') as f:
            pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                     costnode_0, weights_0], f)
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

In [ ]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

In [ ]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

In [ ]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    if counter > 20:
        break
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        
        if conv_1[i] == [True, True]:
            continue
            
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        if not type(bestControl_1[i]) == type(None):
            control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1].copy()
        else:
            control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1].copy()
            cost_1[i] = cost_0[i]
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        with open(final_file_1,'wb') as f:
            pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)
            
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
    counter += 1